![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)







# Import in apple verion
### Check Critical Package Version
✅ JAX: 0.6.2
✅ MuJoCo: 3.3.6
✅ Brax: 0.13.0
✅ Flax: 0.10.7

In [ ]:
from apple_mujoco_setup import *

# UR10e with Hand E - Training with Weights and Biases 

1. Load Model

In [ ]:
env_name = 'UR10PickCube'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)

2. get training params and change them

In [ ]:
# Start from original DM config
base_ppo_params = manipulation_params.brax_ppo_config(env_name)
ppo_params = deepcopy(base_ppo_params)

# Example: keep original config for a "real" run
# (if you want a tiny smoke-test, override here like before)

# Create a seed and keep it
import numpy as np
seed = int(np.random.randint(0, 2**31 - 1))

# Convert ppo_params to a flat dict for wandb.config
# ml_collections ConfigDict → plain dict
try:
    ppo_cfg_dict = ppo_params.to_dict()
except AttributeError:
    ppo_cfg_dict = dict(ppo_params)


## Init Weights&Biases

💡 What gets logged as config (hyperparameters): 

- env_name, algo, seed 
- For PPO: 

    - num_timesteps, num_envs, episode_length, unroll_length, batch_size, num_minibatches, learning_rate, entropy_cost, discounting, etc.

In [ ]:
wandb_cfg = {
    "env_name": env_name,
    "algo": "PPO",
    "seed": seed,
}
for k, v in ppo_cfg_dict.items():
    if isinstance(v, (int, float, str, bool)):
        wandb_cfg[f"ppo/{k}"] = v

run = wandb.init(
    project="ur10_ur10pickcube",    # << change to your W&B project
    job_type="train",
    config=wandb_cfg,
)

## Training Algorithm

Good metrics & hyperparams to log (most are already in metrics / config):

Hyperparams (wandb.config):

env_name, seed

PPO: num_timesteps, num_envs, episode_length, unroll_length, batch_size, num_minibatches, learning_rate, entropy_cost, discounting, gae_lambda, clipping_epsilon, max_grad_norm, etc.

Metrics (wandb.log via progress_wandb):

eval/episode_reward, eval/episode_reward_std

train/episode_reward (if Brax exposes it)

loss/policy, loss/value, loss/entropy

learning_rate

Env-specific metrics if exposed: gripper_box, box_target, no_floor_collision, robot_target_qpos, etc.

In [ ]:
def progress_wandb(num_steps, metrics):
    """
    Called periodically by ppo.train.
    Logs scalar metrics to W&B at the given step.
    """
    log_dict = {"training/num_steps": num_steps}
    for k, v in metrics.items():
        try:
            log_dict[k] = float(v)
        except Exception:
            # ignore non-scalars
            continue

    wandb.log(log_dict, step=num_steps)


## Rollout + video logging helper

We’ll use a separate env for rollouts so we don’t disturb training state.
Adjust the render line if your env uses a different interface.

In [ ]:
def rollout_and_log_video(
    num_steps,
    make_policy,
    params,
    env_name,
    video_length=200,
    fps=30,
    camera_kwargs=None,
    step_tag=None,
):
    """
    Runs a rollout with the current policy and logs a video to W&B.
    Called from policy_params_fn during training.

    num_steps: current training step (for logging)
    make_policy: function from params -> policy_fn
    params: current PPO params
    env_name: name for registry.load
    """
    camera_kwargs = camera_kwargs or {}

    # Fresh env for visualization
    env = registry.load(env_name)

    # Build policy_fn from params
    policy_fn = make_policy(params)

    key = jax.random.PRNGKey(seed + 123)  # deterministic but separate from training
    state = env.reset(key)

    frames = []
    max_steps = video_length

    for t in range(max_steps):
        obs = state.obs
        key, sk = jax.random.split(key)
        # Adapt this line if your policy signature differs
        action, _ = policy_fn(obs, sk)

        state = env.step(state, action)

        # Adjust this line if your env's render has a different API
        frame = env.render(mode="rgb_array", **camera_kwargs)
        frames.append(frame)

        if bool(state.done):
            break

    # Save to disk under this run directory
    tag = step_tag if step_tag is not None else f"{num_steps}"
    video_filename = f"rollout_step_{tag}.mp4"
    video_path = os.path.join(wandb.run.dir, video_filename)

    imageio.mimsave(video_path, frames, fps=fps)

    wandb.log(
        {"rollout/video": wandb.Video(video_path, fps=fps, format="mp4")},
        step=num_steps,
    )

    return video_path



## Save the trained model to W&B
After training returns params, save them and attach as a W&B artifact.
Now the model is stored inside the W&B run and as an artifact you can fetch later.

In [ ]:
def policy_params_wandb(num_steps, make_policy, params):
    """
    Called by ppo.train with the latest policy params at some intervals.
    We use this to:
      - log a rollout video to W&B every VIDEO_EVERY_STEPS environment steps.
    """
    # Store latest step/params in case you also want to save mid-training models
    wandb.run.summary["latest_num_steps"] = int(num_steps)

    # Decide whether to log a video at this step
    if num_steps >= _video_state["next_video_step"]:
        print(f"[policy_params_fn] Logging video at step {num_steps}...")
        rollout_and_log_video(
            num_steps=num_steps,
            make_policy=make_policy,
            params=params,
            env_name=env_name,
            video_length=200,
            fps=30,
            camera_kwargs={},          # e.g. {"camera_id": 0}
            step_tag=num_steps,
        )
        _video_state["next_video_step"] += _video_state["video_every"]


## Build and run PPO training with both hooks

In [ ]:
ppo_training_params = dict(ppo_params)

network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
    del ppo_training_params["network_factory"]
    network_factory = functools.partial(
        ppo_networks.make_ppo_networks,
        **ppo_params.network_factory,
    )

train_fn = functools.partial(
    ppo.train,
    **ppo_training_params,
    network_factory=network_factory,
    progress_fn=progress_wandb,           # logs metrics to W&B
    policy_params_fn=policy_params_wandb, # logs videos + hooks into params
    seed=seed,
)

make_inference_fn, params, final_metrics = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
)

## Save final model as a W&B artifact

In [ ]:
model_path = os.path.join(wandb.run.dir, "ur10_ppo_params_final.pkl")

to_save = {
    "params": params,
    "seed": seed,
    "env_name": env_name,
    "ppo_config": ppo_cfg_dict,
    "timestamp": datetime.now().isoformat(),
}

with open(model_path, "wb") as f:
    pickle.dump(to_save, f)

artifact = wandb.Artifact("ur10_ppo_policy", type="model")
artifact.add_file(model_path)
wandb.log_artifact(artifact)

# Optionally mark some summary stats
if "eval/episode_reward" in final_metrics:
    wandb.run.summary["final_eval_return"] = float(final_metrics["eval/episode_reward"])
